<a href="https://colab.research.google.com/github/shaestasaleem/flyrank-machine-learning-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaestasaleem/flyrank-machine-learning-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My selected lane is Refresh / Content Opportunity Scoring.

The primary machine learning task is ranking or scoring. The system will assign a review priority score to each content page and rank the pages that should be reviewed first.

A classification model may estimate the probability of an observed decline, but the final output will be a ranked list because the content team has limited time and cannot review every page.

In [8]:
import pandas as pd

data_url = "https://raw.githubusercontent.com/shaestasaleem/flyrank-machine-learning-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_url)

print("Dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully
Rows: 30000
Columns: 44


## 2. Target or proxy

My provisional target is is_declining_label.

The original dataset does not contain this target as a separate column, so I create it from the observed trend_direction field. Pages with a downward trend are labelled 1, while the remaining pages are labelled 0.

This label is a proxy for review priority. It does not mean that every declining page definitely needs editing or that editing the page will improve its ranking.

The trend_direction and trend_pct columns will not be used as model input features because they are directly related to the target and would cause data leakage.

In [9]:

df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Target column created: is_declining_label")
print()

print(df["is_declining_label"].value_counts())

declining_share = df["is_declining_label"].mean() * 100

print()
print("Observed declining-page share:", round(declining_share, 2), "%")

Target column created: is_declining_label

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Observed declining-page share: 54.21 %


## 3. Success metric
My primary success metric will be Precision@50.

Precision@50 measures how many of the top 50 pages recommended by the system are observed declining cases. This metric matches the real action because an editor may only have enough time to review a limited number of pages.

My provisional definition of good performance is Precision@50 of at least 0.60. This would mean that at least 30 of the top 50 recommendations are relevant observed cases.

The result should also perform better than a transparent fixed-rule baseline.

In [10]:
k = 50
minimum_precision = 0.60
minimum_relevant_pages = int(k * minimum_precision)

print("Primary metric: Precision@50")
print("Review capacity:", k, "pages")
print("Provisional good score:", minimum_precision)
print(
    "This means at least",
    minimum_relevant_pages,
    "of the top",
    k,
    "pages should be relevant observed cases."
)

Primary metric: Precision@50
Review capacity: 50 pages
Provisional good score: 0.6
This means at least 30 of the top 50 pages should be relevant observed cases.


## 4. The unit of analysis, as a real dataframe
The unit of analysis is one anonymized content page.

Each dataframe row represents one content page. The columns contain page-level information such as impressions, average ranking position, click-through rate, content age, client identifier, and the observed decline label.

The content_id and client_id columns are identifiers. They may be used for grouping and validation, but they will not be used as predictive model features.

In [11]:
display_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "is_declining_label"
]

page_level_df = df[display_columns].copy()

print("One row represents one content page.")
print("Shape of page-level dataframe:", page_level_df.shape)

page_level_df.head(10)

One row represents one content page.
Shape of page-level dataframe: (30000, 7)


,content_id,client_id,impressions_90d,avg_position,ctr,content_age_days,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,187,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,445,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,141,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,463,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,263,1
5,content_d4084a4bc775,client_f369cb89fc,3970,8.5,0.03,147,1
6,content_9a34b442b552,client_8722616204,20,7.0,0.00,90,1
7,content_a63219c6e95a,client_19581e27de,1724,21.2,0.06,445,0
8,content_5e6c160719bc,client_6208ef0f77,32574,46.0,0.09,90,1
9,content_c27558df2b0c,client_19581e27de,1240,4.9,0.16,257,1


## 5. Why ML beats a fixed rule here
The simple fixed rule recommended 5,335 pages, but only 2,279 of them were observed declining cases. This means the rule selected many pages that were not declining, while it also missed 13,983 declining pages.

These measured results suggest that the fixed rule is too limited. A machine learning model may produce a more useful ranking by considering several page-level signals together.

In [12]:
df["fixed_rule_flag"] = (
    (df["content_age_days"] >= 365)
    & (df["impressions_90d"] >= 100)
).astype(int)

rule_comparison = pd.crosstab(
    df["fixed_rule_flag"],
    df["is_declining_label"],
    rownames=["Fixed rule recommendation"],
    colnames=["Observed decline label"]
)

print("Comparison of a simple fixed rule with the observed target:")
rule_comparison

Comparison of a simple fixed rule with the observed target:


Observed decline label,0,1
Fixed rule recommendation,,
0,10682,13983
1,3056,2279


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.